# kaggle-llm-server — Run All (Single GPU LLM Server)

**Перед запуском:** Settings → Accelerator → **GPU T4 x2** (или x1), Internet → **On**.

Этот ноутбук: 1) распаковывает проект, 2) проверяет железо, 3) ставит зависимости,
4) собирает llama.cpp с CUDA, 5) скачивает GGUF модель, 6) подбирает оптимальные параметры,
7) запускает сервер и туннель на GPU 0, 8) запускает Telegram-бота в фоне.

Просто нажмите **Run All**. В конце в выводе появятся публичные ссылки.

In [ ]:
!nvidia-smi

## 1. Получение исходников проекта из GitHub

Ноутбук при каждом запуске получает актуальную версию из репозитория **PoFigiHubIO/KeglaAI**.
Репозиторий публичный, поэтому токен GitHub в Kaggle не требуется. Исходники копируются в
`/kaggle/working`, а модели и готовая сборка сохраняются между перезапусками сессии.

In [ ]:
import os, shutil, subprocess

REPO_URL = "https://github.com/PoFigiHubIO/KeglaAI.git"
BRANCH = "main"
PROJECT_SUBDIR = "kaggle-llm-server"
REPO_DIR = "/kaggle/working/KeglaAI"
WORK_DIR = f"{REPO_DIR}/{PROJECT_SUBDIR}"

# Клонируем при первом запуске; при повторном — получаем свежую версию кода.
# Не используем git clean: models/, llama.cpp/ и logs/ остаются в Persistence.
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH, "--depth", "1"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

assert os.path.isfile(os.path.join(WORK_DIR, "start.py")), f"Проект не найден: {WORK_DIR}"
os.chdir(WORK_DIR)
print("Рабочая директория:", os.getcwd())
print("Версия исходников:", subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], text=True).strip())


from kaggle_secrets import UserSecretsClient
import os
try:
    os.environ["NGROK_AUTHTOKEN"] = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
    print("Добавлен токен NGROK_AUTHTOKEN")
except Exception as e:
    print("Предупреждение: NGROK_AUTHTOKEN не найден в Secrets. Если используется Ngrok, туннель может не запуститься.")


In [ ]:
from kaggle_secrets import UserSecretsClient
try:
    os.environ["NGROK_AUTHTOKEN"] = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
    print("Добавлен токен NGROK_AUTHTOKEN")
except Exception as e:
    print("Предупреждение: NGROK_AUTHTOKEN не найден в Secrets. Если используется Ngrok, туннель может не запуститься.")

try:
    os.environ["NGROK_AUTHTOKEN_2"] = UserSecretsClient().get_secret("NGROK_AUTHTOKEN_2")
    print("Добавлен токен NGROK_AUTHTOKEN_2")
except Exception as e:
    print("Предупреждение: NGROK_AUTHTOKEN_2 не найден в Secrets. Если используется Ngrok, туннель для второй модели может не запуститься.")


import os
WORK_DIR = '/kaggle/working/KeglaAI/kaggle-llm-server'
assert os.path.isfile(os.path.join(WORK_DIR, 'start.py')), 'Сначала выполните ячейку 1: получение исходников из GitHub.'
os.chdir(WORK_DIR)

print('=== ЗАПУСК LLM СЕРВЕРА И ТЕЛЕГРАМ-БОТА (GPU 0, порт 8080) ===')
!python start.py


In [ ]:
import os
WORK_DIR = '/kaggle/working/KeglaAI/kaggle-llm-server'
assert os.path.isfile(os.path.join(WORK_DIR, 'start.py')), 'Сначала выполните ячейку 1: получение исходников из GitHub.'
os.chdir(WORK_DIR)

print('=== ЗАПУСК ПЕРВОЙ МОДЕЛИ (GPU 0, порт 8080) ===')
!CONFIG_FILE=config_gpu0.yaml python start.py

print('\n=== ЗАПУСК ВТОРОЙ МОДЕЛИ (GPU 1, порт 8081) ===')
!CONFIG_FILE=config_gpu1.yaml python start.py


import json, os
WORK_DIR = '/kaggle/working/KeglaAI/kaggle-llm-server'

params_file = os.path.join(WORK_DIR, 'logs/optimized_params_8080.json')
if os.path.isfile(params_file):
    print('\n=== Параметры для порта 8080 ===')
    with open(params_file) as f:
        print(json.dumps(json.load(f), indent=2, ensure_ascii=False))
else:
    print('\nПараметры для порта 8080 не найдены.')

print('\n=== Проверка доступности моделей и сервисов (локально) ===')
print('Порт 8080 (LLM API):')
!curl -s http://127.0.0.1:8080/v1/models


In [ ]:
import json, os
WORK_DIR = '/kaggle/working/KeglaAI/kaggle-llm-server'

for port in [8080, 8081]:
    params_file = os.path.join(WORK_DIR, f'logs/optimized_params_{port}.json')
    if os.path.isfile(params_file):
        print(f'\n=== Параметры для порта {port} ===')
        with open(params_file) as f:
            print(json.dumps(json.load(f), indent=2, ensure_ascii=False))
    else:
        print(f'\nПараметры для порта {port} не найдены.')

print('\n=== Проверка доступности моделей и сервисов (локально) ===')
print('Порт 8080 (LLM API):')
!curl -s http://127.0.0.1:8080/v1/models
print('\nПорт 8081 (Media Server Health & Status):')
!curl -s http://127.0.0.1:8081/health
print('\n')
!curl -s http://127.0.0.1:8081/status


## 5. Keep-alive

Эта ячейка держит сессию Kaggle активной, пока серверы работают.
Остановите ячейку (interrupt), когда закончите пользоваться сервером.

In [ ]:
import time, os
print("Оба сервера запущены и доступны по публичным ссылкам, показанным выше.")
print("Оставьте эту ячейку выполняться, чтобы сессия не завершилась.")
try:
    while not os.path.exists("/tmp/handover_complete"):
        time.sleep(10)
    print("Передача управления завершена! Завершаем сессию Kaggle...")
except KeyboardInterrupt:
    print("Остановлено пользователем.")